In [1]:
%load_ext autoreload
%autoreload 2
%time

CPU times: user 3 µs, sys: 2 µs, total: 5 µs
Wall time: 6.68 µs


In [2]:
import sys, os
sys.path.append("../../")  # if needed
import pandas as pd
import data_loading as dl

from numu_tki.cc0pi_analyzer import apply_ccnp0pi_stv

In [3]:
RUN = ["1"]
blinded = True

rundata, mc_weights, data_pot = dl.load_runs(
    RUN,
    data="bnb",
    loadpi0variables=False,
    loadshowervariables=True,
    loadrecoveryvars=False,
    loadsystematics=True,
    numupresel=False,
    loadnumuvariables=True,
    use_bdt=True,
    load_lee=False,
    load_nue_tki=False,
    load_numu_tki=True,
    blinded=blinded,
    enable_cache=True,
)


Loading run 1


In [4]:
import xgboost as xgb
MODEL = "/exp/uboone/app/users/rlalnunt/PELEE_BDT/cc0pi_fstrack_pid_softprob.json"
assert os.path.isfile(MODEL), f"Model file not found: {MODEL}"
print("Using model:", MODEL)
print("xgboost version:", xgb.__version__)

Using model: /exp/uboone/app/users/rlalnunt/PELEE_BDT/cc0pi_fstrack_pid_softprob.json
xgboost version: 1.6.2


In [5]:
print(rundata.keys())
mc = rundata["mc"].copy()
print(mc)

dict_keys(['data', 'ext', 'mc', 'nue', 'drt'])
         shr_hits_y_tot  nproton  shrclusfrac1  mcf_npm  shr_bkt_pdg  \
entry                                                                  
0                     0        0 -3.402823e+38        0            0   
1                     0        1 -3.402823e+38        0            0   
2                     0        0 -3.402823e+38        0            0   
3                     0        1 -3.402823e+38        0            0   
4                     0        0 -3.402823e+38        0            0   
...                 ...      ...           ...      ...          ...   
1025780               0        0 -3.402823e+38        0            0   
1025781               0        1 -3.402823e+38        1            0   
1025782               0        2 -3.402823e+38        0            0   
1025783             243        0  9.245283e-01        0           13   
1025784               0        0 -3.402823e+38        0            0   

         shrsubc

In [ ]:
from numu_tki import cc0pi_analyzer
mc = apply_ccnp0pi_stv(mc, model_path=MODEL)
#print(mc)
print(mc.shape)      # or:
mc.head(3)

In [ ]:
# --- HOT PATCH: create any columns the current checker expects but your ntuple lacks ---
import numpy as np
from numu_tki import cc0pi_analyzer as a

def _like_gen(gen_col, filler):
    n = len(gen_col)
    return filler(n)

# If the loader didn’t bring these in, synthesize them:
if 'n_tracks' not in mc.columns:
    mc['n_tracks'] = mc['trk_len_v'].apply(lambda v: len(v))

if 'sel_reco_vertex_in_FV' not in mc.columns:
    mc['sel_reco_vertex_in_FV'] = mc.apply(
        lambda r: a.in_proton_containment_vol(
            float(r['reco_nu_vtx_sce_x']),
            float(r['reco_nu_vtx_sce_y']),
            float(r['reco_nu_vtx_sce_z'])
        ), axis=1
    )

# Vector branches the Python expects (make safe defaults if missing)
for name, make in [
    ('pfp_trk_daughters_v',       lambda n: [0]*n),
    ('pfp_shr_daughters_v',       lambda n: [0]*n),
    ('trk_pid_chipr_v',           lambda n: [np.nan]*n),  # optional in logic
    ('trk_pid_chimu_v',           lambda n: [np.nan]*n),  # optional in logic
    ('trk_bragg_mu_fwd_preferred_v', lambda n: [1]*n),    # default so "flipped" veto won’t trigger
]:
    if name not in mc.columns:
        mc[name] = mc['pfp_generation_v'].apply(lambda g: make(len(g)))


In [ ]:
mc = a.apply_ccnp0pi_stv(mc, model_path=MODEL)

In [ ]:
print(mc)

In [ ]:
# 1) Provide the proper FV flag (prefer the branch already in your ntuple)
if 'sel_reco_vertex_in_FV' not in mc.columns:
    if 'isVtxInFiducial' in mc.columns:
        mc['sel_reco_vertex_in_FV'] = mc['isVtxInFiducial'].astype(bool)
    else:
        # absolute fallback if that branch truly doesn't exist:
        # mark as False when any vtx coordinate is the sentinel
        import numpy as np
        for k in ['reco_nu_vtx_sce_x','reco_nu_vtx_sce_y','reco_nu_vtx_sce_z']:
            mc[k] = mc[k].replace(-3.402823e+38, np.nan)
        # loose FV (you can tighten if you know the exact FV)
        mc['sel_reco_vertex_in_FV'] = (
            mc['reco_nu_vtx_sce_x'].between(0, 256) &
            mc['reco_nu_vtx_sce_y'].between(-116, 116) &
            mc['reco_nu_vtx_sce_z'].between(0, 1036)
        ).fillna(False)

# 2) Make sure the “missing” vector branches exist (harmless defaults if your loader didn’t read them)
import numpy as np
def _ensure_vec(col, filler):
    if col not in mc.columns:
        mc[col] = mc['pfp_generation_v'].apply(lambda g: filler(len(g)))

_ensure_vec('pfp_trk_daughters_v',       lambda n: [0]*n)
_ensure_vec('pfp_shr_daughters_v',       lambda n: [0]*n)
_ensure_vec('trk_pid_chipr_v',           lambda n: [np.nan]*n)  # optional for BDT
_ensure_vec('trk_pid_chimu_v',           lambda n: [np.nan]*n)  # only used in a niche veto
_ensure_vec('trk_bragg_mu_fwd_preferred_v', lambda n: [1]*n)    # default disables “flipped track” veto

# 3) Also ensure we have a track count (C++ uses num_tracks)
if 'n_tracks' not in mc.columns:
    mc['n_tracks'] = mc['trk_len_v'].apply(len)


In [ ]:
import numpy as np
import pandas as pd

# 0) Work on a copy
mc = mc.copy()

# 1) Drop duplicate columns (keep the first occurrence)
dups = mc.columns[mc.columns.duplicated()]
if len(dups):
    print("Dropping duplicate columns:", sorted(set(dups)))
mc = mc.loc[:, ~mc.columns.duplicated()]

# 2) Ensure the FV flag is a scalar bool per row (matches the C++: point_inside_FV)
# Prefer the existing branch if present
if "sel_reco_vertex_in_FV" not in mc.columns:
    if "isVtxInFiducial" in mc.columns:
        mc["sel_reco_vertex_in_FV"] = mc["isVtxInFiducial"].astype(bool)
    else:
        # Fallback: compute a loose FV from coordinates; ignore the sentinel -3.402e38
        for k in ["reco_nu_vtx_sce_x","reco_nu_vtx_sce_y","reco_nu_vtx_sce_z"]:
            if k in mc.columns:
                mc[k] = mc[k].replace(-3.402823e38, np.nan)
        mc["sel_reco_vertex_in_FV"] = (
            mc["reco_nu_vtx_sce_x"].between(0, 256) &
            mc["reco_nu_vtx_sce_y"].between(-116, 116) &
            mc["reco_nu_vtx_sce_z"].between(0, 1036)
        ).fillna(False)

# 3) Ensure we have n_tracks (the C++ uses num_tracks)
if "n_tracks" not in mc.columns:
    mc["n_tracks"] = mc["trk_len_v"].apply(lambda v: len(v) if isinstance(v, (list, tuple, np.ndarray)) else 0)

# 4) Normalize any vector-like columns so each cell is a Python list
def _to_list_cell(x):
    if isinstance(x, list): return x
    if isinstance(x, tuple): return list(x)
    if isinstance(x, np.ndarray): return x.tolist()
    if isinstance(x, pd.Series): return x.tolist()
    return x  # leave scalars as-is

vec_cols = [
    "pfp_generation_v","trk_score_v","trk_distance_v","trk_len_v",
    "trk_llr_pid_score_v","trk_energy_proton_v",
    "trk_dir_x_v","trk_dir_y_v","trk_dir_z_v",
    "trk_sce_end_x_v","trk_sce_end_y_v","trk_sce_end_z_v",
    "trk_sce_start_x_v","trk_sce_start_y_v","trk_sce_start_z_v",
    "trk_range_muon_mom_v","trk_mcs_muon_mom_v",
    # optional but referenced:
    "pfp_trk_daughters_v","pfp_shr_daughters_v",
    "trk_pid_chipr_v","trk_pid_chimu_v","trk_bragg_mu_fwd_preferred_v",
]

for c in vec_cols:
    if c in mc.columns:
        mc[c] = mc[c].apply(_to_list_cell)

# 5) Provide harmless defaults for truly-missing optional branches (C++ guards these)
def _ensure_vec(col, default_factory):
    if col not in mc.columns:
        mc[col] = mc["pfp_generation_v"].apply(lambda g: default_factory(len(g) if isinstance(g, list) else 0))

_ensure_vec("pfp_trk_daughters_v",          lambda n: [0]*n)
_ensure_vec("pfp_shr_daughters_v",          lambda n: [0]*n)
_ensure_vec("trk_pid_chipr_v",              lambda n: [np.nan]*n)  # not required by BDT
_ensure_vec("trk_pid_chimu_v",              lambda n: [np.nan]*n)  # only used in a narrow veto
_ensure_vec("trk_bragg_mu_fwd_preferred_v", lambda n: [1]*n)       # default disables “flipped” veto


In [ ]:
from numu_tki import cc0pi_analyzer as a
mc = a.apply_ccnp0pi_stv(mc, model_path=MODEL)
print(mc[[
    "sel_reco_vertex_in_FV","sel_pfp_starts_in_PCV","sel_topo_cut_passed",
    "sel_no_reco_showers","sel_presel","sel_has_muon_candidate",
    "sel_nu_mu_cc","sel_CCNp0pi","sel_CC0pi","sel_CC0pi_wc"
]].mean())


In [ ]:
# keep the last occurrence of any duplicated column name
mc = mc.loc[:, ~mc.columns.duplicated(keep='last')]


In [ ]:
print(mc[[
    "sel_reco_vertex_in_FV","sel_pfp_starts_in_PCV","sel_topo_cut_passed",
    "sel_no_reco_showers","sel_presel","sel_has_muon_candidate",
    "sel_nu_mu_cc","sel_CCNp0pi","sel_CC0pi","sel_CC0pi_wc"
]].mean())


In [ ]:
def apply_ccnp0pi_stv(df: pd.DataFrame, model_path: str) -> pd.DataFrame:
    booster = _load_booster(model_path)

    required = [
        "pfp_generation_v","trk_score_v","trk_distance_v","trk_len_v",
        "trk_llr_pid_score_v","trk_energy_proton_v",
        "trk_dir_x_v","trk_dir_y_v","trk_dir_z_v",
        "trk_sce_end_x_v","trk_sce_end_y_v","trk_sce_end_z_v",
        "trk_sce_start_x_v","trk_sce_start_y_v","trk_sce_start_z_v",
        "trk_range_muon_mom_v","trk_mcs_muon_mom_v",
        "topological_score","CosmicIP","reco_nu_vtx_sce_x","reco_nu_vtx_sce_y","reco_nu_vtx_sce_z",
        "nslice","n_tracks"
    ]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise KeyError(f"Missing required columns: {missing}")

    out = df.apply(lambda r: analyze_event(r, booster), axis=1, result_type="expand")

    # *** NEW: drop any columns that we’re about to (re)create ***
    cols_to_replace = list(out.columns)
    base = df.drop(columns=[c for c in cols_to_replace if c in df.columns], errors='ignore')

    return pd.concat([base.reset_index(drop=True), out.reset_index(drop=True)], axis=1)


In [ ]:
import numpy as np

def debug_bdt_inputs(df):
    # pick a preselected event
    rows = df.index[df["sel_presel"] == True]
    if len(rows) == 0:
        print("No events passed presel.")
        return
    r = df.loc[rows[0]]

    gen   = list(r["pfp_generation_v"])
    tscore= list(r["trk_score_v"])
    prange= list(r["trk_range_muon_mom_v"])
    pmcs  = list(r["trk_mcs_muon_mom_v"])
    trk_dist = list(r["trk_distance_v"])
    pid_score = list(r["trk_llr_pid_score_v"])
    KE_p = list(r["trk_energy_proton_v"])
    endx = list(r["trk_sce_end_x_v"]); endy=list(r["trk_sce_end_y_v"]); endz=list(r["trk_sce_end_z_v"])
    n_trk = list(r.get("pfp_trk_daughters_v", [0]*len(gen)))
    n_shr = list(r.get("pfp_shr_daughters_v", [0]*len(gen)))
    chi2p = list(r.get("trk_pid_chipr_v", [np.nan]*len(gen)))

    n = len(gen)
    ok=0; bad=0
    for i in range(n):
        if int(gen[i])!=2 or float(tscore[i])<=0.5: continue
        trk_cont = (10.0 < float(endx[i]) < 246.35) and (-106.5 < float(endy[i]) < 106.5) and (10.0 < float(endz[i]) < 1026.8)
        rel = (pmcs[i]-prange[i])/prange[i] if (np.isfinite(prange[i]) and prange[i]!=0) else np.nan
        fs = [trk_dist[i], pid_score[i], chi2p[i], KE_p[i], float(trk_cont), float(n_trk[i]+n_shr[i]), rel]
        if all(np.isfinite(v) for v in fs):
            ok += 1
        else:
            bad += 1
    print(f"Gen==2 & trk_score>0.5 candidates: valid feature rows={ok}, dropped={bad}")

debug_bdt_inputs(mc)



In [ ]:
import numpy as np

def explain_drops(df, n_events=10):
    checked = 0
    for idx, r in df[df["sel_presel"] == True].head(n_events).iterrows():
        gen   = list(r["pfp_generation_v"])
        tscore= list(r["trk_score_v"])
        prange= list(r["trk_range_muon_mom_v"])
        pmcs  = list(r["trk_mcs_muon_mom_v"])
        trk_dist = list(r["trk_distance_v"])
        pid_score = list(r["trk_llr_pid_score_v"])
        KE_p = list(r["trk_energy_proton_v"])
        endx = list(r["trk_sce_end_x_v"]); endy=list(r["trk_sce_end_y_v"]); endz=list(r["trk_sce_end_z_v"])
        n_trk = list(r.get("pfp_trk_daughters_v", [0]*len(gen)))
        n_shr = list(r.get("pfp_shr_daughters_v", [0]*len(gen)))
        chi2p = list(r.get("trk_pid_chipr_v", [np.nan]*len(gen)))

        for i in range(len(gen)):
            if int(gen[i])!=2 or float(tscore[i])<=0.5: continue
            rel = (pmcs[i]-prange[i])/prange[i] if (np.isfinite(prange[i]) and prange[i]!=0) else np.nan
            fields = {
                "trk_distance_v": trk_dist[i],
                "trk_llr_pid_score_v": pid_score[i],
                "trk_pid_chipr_v": chi2p[i],
                "trk_energy_proton_v": KE_p[i],
                "trk_contained_flag": float(10.0 < endx[i] < 246.35 and -106.5 < endy[i] < 106.5 and 10.0 < endz[i] < 1026.8),
                "n_daughters": float(n_trk[i] + n_shr[i]),
                "rel_mcs_range": rel,
            }
            bad = {k:v for k,v in fields.items() if not np.isfinite(v)}
            if bad:
                print(f"[event {idx}] dropped track {i} because non-finite: {bad}")
            else:
                print(f"[event {idx}] track {i} would be VALID")
            checked += 1
    if checked == 0:
        print("No candidate tracks found in the first few preselected events.")

explain_drops(mc)
